# XGBoost

**The big picture:** the data has 683 breast-tissue samples. Each one has 9 scores from 1 to 10 describing how the cells look under a microscope (size, shape, thickness and so on). The goal is to predict whether the tumour is **benign** (harmless) or **malignant** (cancer).

**XGBoost** builds **many small decision trees, each one fixing the previous ones' mistakes**. The first tree makes rough guesses. The second tree focuses on where the first one went wrong, the third on what is still wrong, and so on. Added together, lots of weak guessers become one strong one. This step-by-step fixing is called *gradient boosting*. XGBoost ("eXtreme Gradient Boosting") is a fast, popular version of it that often wins competitions on table-shaped data.

## Importing the libraries

- **pandas** reads and handles tables, like Excel in Python.
- **numpy** does fast maths on lists of numbers.
- **matplotlib** draws charts. It is imported out of habit but not used: this notebook has no plots.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

- Reads the CSV file into a table of **683 samples**.
- **X** = every column except the last: *Sample code number*, then the 9 cell scores (Clump Thickness, Uniformity of Cell Size, Uniformity of Cell Shape, Marginal Adhesion, Single Epithelial Cell Size, Bare Nuclei, Bland Chromatin, Normal Nucleoli, Mitoses).
- **y** = *Class*: **2 = benign** (444 samples), **4 = malignant** (239 samples). This is what we want to **predict**.

Note that *Sample code number* is just an ID label, so it doesn't really describe the cells. The code keeps it in X anyway; dropping it would be cleaner.

In [0]:
dataset = pd.read_csv('Data.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Splitting the dataset into the Training set and Test set

- **80% of the samples (546)** go into the *training set*. The model learns from these.
- **20% (137 samples)** go into the *test set*, held back to check the model afterwards.
- `random_state = 0` makes the shuffle come out the same every time.

There is **no feature scaling** in this notebook. **Why?** Decision trees only ask yes/no questions like "is Clump Thickness above 5?". The size of the numbers doesn't matter to them, so scaling isn't needed.

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Training XGBoost on the Training set

- `XGBClassifier()` creates the model with all **default settings** (for example, 100 trees, each limited to a handful of questions deep).
- `.fit()` builds the trees one after another, each one trained to correct the mistakes left by the trees before it.

**Heads-up:** newer versions of XGBoost expect the classes to be numbered 0 and 1. Because this data uses 2 and 4, `.fit()` may stop with an error saying it expected `[0 1]` but got `[2 4]`. If that happens, turn the labels into 0/1 first (for example with scikit-learn's `LabelEncoder`). The notebook has no saved output, so there are no results to read here yet.

In [0]:
from xgboost import XGBClassifier
classifier = XGBClassifier()
classifier.fit(X_train, y_train)

## Making the Confusion Matrix

- `predict` guesses benign / malignant for the 137 test samples.
- The **confusion matrix** is a 2×2 table: rows = the real class (2 then 4), columns = the guessed class (2 then 4).
  - Top-left: benign correctly called benign.
  - Bottom-right: malignant correctly called malignant.
  - Top-right: benign wrongly called malignant (a false alarm).
  - Bottom-left: malignant wrongly called benign (a **missed cancer**, the most serious mistake here).
- `accuracy_score` = correct guesses ÷ 137.

The notebook has no saved output, so run the cell to see the numbers. XGBoost usually scores very highly on this dataset.

In [0]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Applying k-Fold Cross Validation

- A single test score can be lucky or unlucky, so the model also gets **10 mini exams**.
- `cv = 10` cuts the **546 training samples into 10 groups** (about 55 each). The model is trained 10 times, each time tested on a different group and trained on the other 9.
- The first print is the **average accuracy** of the 10 exams: the best estimate of how good the model really is.
- The second print is the **standard deviation**: how much the 10 scores jump around. Small = consistently good; large = the result depends a lot on which samples it sees.

**In one sentence:** load 683 breast-tissue samples, train XGBoost (many small trees, each fixing the previous ones' mistakes) to tell benign from malignant, then check it with a confusion matrix on 137 unseen samples and with 10-fold cross validation.

In [0]:
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))